# Note: this is just a file to manually test the model's results and isn't used in the rest of the project

In [13]:
import cv2
from PIL import Image

import torch
import torchvision
import torchvision.transforms as transforms
import lightning.pytorch as pl

from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# The actual network. It's essentially a wrapper for the faster rcnn network
class RCNN(pl.LightningModule):
    def __init__(self,
                 num_classes: int,
                 **kwargs):
        super().__init__()

        # Use DEFAULT weights explicitly
        self.rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights)

        in_features = self.rcnn.roi_heads.box_predictor.cls_score.in_features
        self.rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    def forward(
        self,
        input_images: list[torch.Tensor],
        input_data: list[dict[str, torch.Tensor]] | None = None
    ):
        return self.rcnn(input_images, input_data)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

        return [optimizer], [scheduler]

    def training_step(self, train_batch, batch_idx):
        imgs, annotations_batch = train_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('train_loss', loss, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, val_batch, batch_idx):
        imgs, annotations_batch = val_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('val_loss', loss, on_step=True, on_epoch=True)

        return loss

In [ ]:
model = RCNN(num_classes=29).to(device)

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [19]:
model.load_state_dict(torch.load('faster_rcnn_weights.pth'))
model = model.eval()

In [20]:
# Function for the model to make a prediction on an image in BGR format. Returns the predictions
def make_prediction(model, image_bgr):
  #image_bgr = cv2.imread(img_path)
  image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
  image_pil = Image.fromarray(image_rgb)

  # Transform image
  transform = transforms.Compose([transforms.ToTensor()])
                                  
  image_tensor = transform(image_pil).unsqueeze(0) .to(device)
  print(image_tensor.shape)
  model.eval()  # Set the model to evaluation mode
  with torch.no_grad():
      predictions = model(image_tensor.to(device))

  #show_prediction(predictions[0], image_bgr)
  return predictions[0]

In [21]:
# Processess the prediction and returns it in BGR format with all of the bounding boxes and everything drawn on
def process_prediction(prediction, image_bgr):
    boxes = prediction['boxes']
    labels = prediction['labels']
    scores = prediction['scores']


    #label_list = [f"Agent_{i}" for i in range(1, 23)] # Assuming class IDs are 1-22 for agents
    label_list = ["Chamber",
                "Cypher",
                "Yoru",
                "Killjoy",
                "KAYO",
                "Gekko",
                "Astra",
                "Deadlock",
                "Breach",
                "Omen",
                "Reyna",
                "Phoenix",
                "Tejo",
                "Raze",
                "Brimstone",
                "Vyse",
                "Fade",
                "Sova",
                "Jett",
                "Neon",
                "Skye",
                "Viper",
                "Sage"]

    # Threshold
    threshold = 0.5

    # Keep track of whether any boxes were drawn
    b_boxes_drawn = False

    for i in range(len(boxes)):
        if scores[i] > threshold:
            box = boxes[i].cpu().numpy().astype(int)
            label_id = labels[i].item() # Get integer value of label
            # Ensure label_id is within the valid range for label_list
            if 0 < label_id <= len(label_list):
                label = label_list[label_id] # Adjust index if label_list is 0-indexed
            else:
                label = f"Unknown ({label_id})"

            score = scores[i].item()

            # draw label and score
            text = f"{label}: {score:.2f}"
            cv2.putText(image_bgr, text, (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.4, (0, 0, 0), 3, cv2.LINE_AA)
            cv2.putText(image_bgr, text, (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.4, (0, 255, 0), 2, cv2.LINE_AA)

            # Draw rectangle and label
            cv2.rectangle(image_bgr, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
            b_boxes_drawn = True

    # Check if any bounding boxes were drawn, if not, indicate it
    if not b_boxes_drawn:
        return

    return image_bgr

In [ ]:
# Process a video and display it in a window with the bounding boxes and labels
video_path = 'D:/Desktop2/valorant-testing-video-snippet-short.mp4' #CHANGE TO MATCH YOUR PATH
cap = cv2.VideoCapture(video_path)
count = 0
every_n_frames = 30 # How often the model should process a frame

while cap.isOpened():
    ret,frame = cap.read()

    if count % every_n_frames == 0 and ret:
        predictions = make_prediction(model, frame)
        processed_frame = process_prediction(predictions, frame)

        if processed_frame is not None:
            cv2.imshow('window-name', processed_frame)
            cv2.imwrite("frames/frame%04d.jpg" % count, processed_frame)
            
        print(count)


    count = count + 1
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows() # destroy all opened windows


torch.Size([1, 3, 720, 1280])
0
torch.Size([1, 3, 720, 1280])
30
torch.Size([1, 3, 720, 1280])
60
torch.Size([1, 3, 720, 1280])
90
torch.Size([1, 3, 720, 1280])
120
torch.Size([1, 3, 720, 1280])
150
torch.Size([1, 3, 720, 1280])
180
torch.Size([1, 3, 720, 1280])
210
torch.Size([1, 3, 720, 1280])
240
